In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cmocean as cmo

# Datasets to compare. Differences are comparison minus reference; use a gmsl 0
# run as the reference to compare a scenario against present-day topography.
reference_file = 'migrated_bathymetry/lmax-1079/bathymetry_gmsl_0p000_frac_0p00_lmax_1079.nc'
comparison_file = 'migrated_bathymetry/lmax-1079/bathymetry_gmsl_72p34_frac_0p900_lmax_1079.nc'

reference = xr.open_dataset(reference_file)
comparison = xr.open_dataset(comparison_file)

# Check whether the reference and comparison datasets share the same lat/lon grid
coords_match = (
    np.array_equal(reference.lat.values, comparison.lat.values) and
    np.array_equal(reference.lon.values, comparison.lon.values)
)


def plot_map(ax, data, cmap, title, label, extent, vmin=None, vmax=None):
    """Stock panel plotter: shared boilerplate for every map in this notebook."""
    ax.set_extent(list(extent), ccrs.PlateCarree())
    im = data.plot.pcolormesh(
        ax=ax, x='lon', y='lat', transform=ccrs.PlateCarree(),
        cmap=cmap, vmin=vmin, vmax=vmax, add_colorbar=False
    )
    #ax.coastlines()
    ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
    ax.set_title(title)
    ax.figure.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, label=label)
    return im


def sym_limit(da):
    """Symmetric vmax (use as vmin=-sym_limit, vmax=sym_limit) for a difference field."""
    return float(np.nanmax(np.abs(da)))

In [ ]:
# Antarctica: reference, comparison and difference rows
extent = (-180, 180, -90, -60)

fig, axes = plt.subplots(
    3, 3,
    figsize=(18, 18),
    subplot_kw={'projection': ccrs.SouthPolarStereo()},
    layout='constrained'
)

# Subset all data to the Southern Hemisphere (south of 60S)
lat_bounds = slice(-90, -60)
ref_ant = reference.sel(lat=lat_bounds)
comp_ant = comparison.sel(lat=lat_bounds)

# ==========================================
# ROW 1: REFERENCE DATA
# ==========================================
plot_map(axes[0, 0], ref_ant['bedrock'], cmo.cm.topo, "Reference Bedrock", "Elevation (m)", extent)
plot_map(axes[0, 1], ref_ant['ice_thickness'], cmo.cm.ice, "Reference Ice Thickness", "Thickness (m)", extent)
plot_map(axes[0, 2], ref_ant['surface'], cmo.cm.topo, "Reference Surface", "Elevation (m)", extent)

# ==========================================
# ROW 2: COMPARISON DATA
# ==========================================
plot_map(axes[1, 0], comp_ant['bedrock'], cmo.cm.topo, "Comparison Bedrock", "Elevation (m)", extent)
plot_map(axes[1, 1], comp_ant['ice_thickness'], cmo.cm.ice, "Comparison Ice Thickness", "Thickness (m)", extent)
plot_map(axes[1, 2], comp_ant['surface'], cmo.cm.topo, "Comparison Surface", "Elevation (m)", extent)

# ==========================================
# ROW 3: DIFFERENCE (COMPARISON - REFERENCE), only if grids line up
# ==========================================
if coords_match:
    bed_diff = comp_ant['bedrock'] - ref_ant['bedrock']
    ice_diff = comp_ant['ice_thickness'] - ref_ant['ice_thickness']
    sur_diff = comp_ant['surface'] - ref_ant['surface']

    plot_map(axes[2, 0], bed_diff, cmo.cm.balance, "Bedrock Difference (Comparison - Reference)",
             "Elevation diff (m)", extent, vmin=-sym_limit(bed_diff), vmax=sym_limit(bed_diff))
    plot_map(axes[2, 1], ice_diff, cmo.cm.delta_r, "Ice Thickness Difference (Comparison - Reference)",
             "Thickness diff (m)", extent, vmin=-sym_limit(ice_diff), vmax=sym_limit(ice_diff))
    plot_map(axes[2, 2], sur_diff, cmo.cm.tarn_r, "Surface Difference (Comparison - Reference)",
             "Elevation diff (m)", extent, vmin=-400, vmax=400)
else:
    print("Lat/lon grids differ between reference and comparison datasets - skipping difference row.")
    for ax in axes[2]:
        ax.set_visible(False)

In [ ]:
# Greenland: reference, comparison and difference rows
extent = (285, 350, 55, 85)

fig, axes = plt.subplots(
    3, 3,
    figsize=(18, 18),
    subplot_kw={'projection': ccrs.LambertAzimuthalEqualArea(central_longitude=-40, central_latitude=72)},
    layout='constrained'
)

# Subset all data to Greenland (roughly 55N-85N, 10W-75W).
# Note: this dataset's lon coordinate runs 0-360 (not -180-180), so
# Greenland's 75W-10W range is expressed as 285E-350E.
lat_bounds = slice(55, 85)
lon_bounds = slice(285, 350)
ref_grl = reference.sel(lat=lat_bounds, lon=lon_bounds)
comp_grl = comparison.sel(lat=lat_bounds, lon=lon_bounds)

# ==========================================
# ROW 1: REFERENCE DATA
# ==========================================
plot_map(axes[0, 0], ref_grl['bedrock'], cmo.cm.topo, "Reference Bedrock", "Elevation (m)", extent)
plot_map(axes[0, 1], ref_grl['ice_thickness'], cmo.cm.ice, "Reference Ice Thickness", "Thickness (m)", extent)
plot_map(axes[0, 2], ref_grl['surface'], cmo.cm.topo, "Reference Surface", "Elevation (m)", extent)

# ==========================================
# ROW 2: COMPARISON DATA
# ==========================================
plot_map(axes[1, 0], comp_grl['bedrock'], cmo.cm.topo, "Comparison Bedrock", "Elevation (m)", extent)
plot_map(axes[1, 1], comp_grl['ice_thickness'], cmo.cm.ice, "Comparison Ice Thickness", "Thickness (m)", extent)
plot_map(axes[1, 2], comp_grl['surface'], cmo.cm.topo, "Comparison Surface", "Elevation (m)", extent)

# ==========================================
# ROW 3: DIFFERENCE (COMPARISON - REFERENCE), only if grids line up
# ==========================================
if coords_match:
    bed_diff = comp_grl['bedrock'] - ref_grl['bedrock']
    ice_diff = comp_grl['ice_thickness'] - ref_grl['ice_thickness']
    sur_diff = comp_grl['surface'] - ref_grl['surface']

    plot_map(axes[2, 0], bed_diff, cmo.cm.balance, "Bedrock Difference (Comparison - Reference)",
             "Elevation diff (m)", extent, vmin=-sym_limit(bed_diff), vmax=sym_limit(bed_diff))
    plot_map(axes[2, 1], ice_diff, cmo.cm.delta_r, "Ice Thickness Difference (Comparison - Reference)",
             "Thickness diff (m)", extent, vmin=-sym_limit(ice_diff), vmax=sym_limit(ice_diff))
    plot_map(axes[2, 2], sur_diff, cmo.cm.tarn_r, "Surface Difference (Comparison - Reference)",
             "Elevation diff (m)", extent, vmin=-sym_limit(sur_diff), vmax=sym_limit(sur_diff))
else:
    print("Lat/lon grids differ between reference and comparison datasets - skipping difference row.")
    for ax in axes[2]:
        ax.set_visible(False)

In [ ]:
comparison